# 02 · Preprocesamiento

**SIRAP** · Proyecto Integrador en IA (MIAR0545) · Eduardo Alejandro Ceballos Jijón · Christian Xavier Valle Maridueña

Prepara el dataset para Ultralytics (copia local, `data.yaml` con rutas absolutas, muestras fijas del preset `rapido`, 1.1), define la normalización de placas ecuatorianas (`XXX-0000`, correcciones por posición, 2.1) con sus pruebas y arma la referencia corregida de SAVH a partir de la revisión de placas (2.2).

> Este notebook es un extracto de `SIRAP_backend.ipynb` (el notebook completo, que es la fuente de verdad). Las celdas y sus salidas son las mismas y conservan la numeración de secciones del notebook completo; la sección 0 se repite para que corra por sí solo en Colab.


## 0. Preparación del entorno

Esta sección prepara Colab: comprueba la GPU, instala librerías, lee Secrets, clona el repositorio y configura rutas. Ejecuta la sección completa al empezar cada sesión; no monta Google Drive.

### GPU
`nvidia-smi` muestra la GPU asignada. Si falla, ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU T4` y guarda. Sin GPU el notebook funciona, pero entrenar y evaluar el detector se vuelve muy lento (el OCR corre en la CPU en cualquier caso).

In [ ]:
# SIRAP repository bootstrap — safe for Colab and local Jupyter
import sys
import subprocess
from pathlib import Path

REPOSITORY_URL = "https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git"
_repo_candidates = [Path("/content/SIRAP_USA_ECUADOR"), Path("/content/SIRAP_USA_ECUADOR_notebook")]
REPO_ROOT = None
for _candidate in _repo_candidates:
    if (_candidate / ".git").is_dir():
        _status = subprocess.run(["git", "-C", str(_candidate), "status", "--porcelain"],
                                 check=True, capture_output=True, text=True).stdout.strip()
        if not _status:
            subprocess.run(["git", "-C", str(_candidate), "pull", "--ff-only", "origin", "main"], check=True)
        REPO_ROOT = _candidate
        break
    if not _candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(_candidate)], check=True)
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise RuntimeError("No se borró ninguna carpeta existente. Revisa /content/SIRAP_USA_ECUADOR y "
                       "/content/SIRAP_USA_ECUADOR_notebook; ambas existen y no son clones Git.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook_io import repository_paths, publish_artifacts
SIRAP_PATHS = repository_paths(REPO_ROOT)
WORK_DIR = SIRAP_PATHS["workspace"]
print("Repositorio:", REPO_ROOT)
print("Métricas:", SIRAP_PATHS["metrics"], "| Figuras:", SIRAP_PATHS["figures"])

In [ ]:
!nvidia-smi

### Dependencias
Instala solo las librerías que Colab no trae, en las versiones con las que se probó el notebook, para que los resultados se puedan reproducir:

- `ultralytics==8.4.177`: entrenamiento, validación e inferencia de YOLO11.
- `trackers==2.6.1`: OC-SORT, el seguidor de placas del pipeline de video.
- `supervision==0.30.9`: detecciones, anotación de video y mAP por tamaño de placa.
- `fast-plate-ocr==1.1.0` y `onnxruntime==1.31.0`: el OCR de placas (`cct-s-v2-global-model`) y el motor que lo corre en la CPU. fast-plate-ocr se instala sin sus dependencias (`--no-deps`): pide `opencv-python-headless`, que podría chocar con el `opencv-python` de Colab (dos paquetes para el mismo `cv2`), y las demás (`numpy`, `pyyaml`, `rich` y `tqdm`) ya vienen en Colab.
- `roboflow==1.7.1`: descarga del dataset.
- `Levenshtein`: distancia entre placas (regla de decisión y CER).

`torch`, `torchvision`, `opencv`, `pandas` y `matplotlib` ya vienen en Colab y no se tocan. Si al terminar Colab pide **reiniciar la sesión**, reinicia y sigue desde la celda siguiente (no hace falta reinstalar).

La celda de importaciones, más abajo, carga todas las librerías y muestra la GPU y las versiones instaladas.

In [ ]:
!pip install -q "ultralytics==8.4.177" "trackers==2.6.1" "supervision==0.30.9" "roboflow==1.7.1" Levenshtein "onnxruntime==1.31.0"
!pip install -q --no-deps "fast-plate-ocr==1.1.0"  # sin dependencias: usa el OpenCV que ya trae Colab

### Llaves de API (*Secrets* de Colab)

Las llaves no se escriben en el notebook: la función `get_secret` las lee del panel 🔑 **Secrets** de Colab (fuera de Colab, de variables de entorno).

| Secret | Para qué | ¿Obligatorio? |
|---|---|---|
| `ROBOFLOW_API_KEY` | Descargar el dataset de placas desde Roboflow (Roboflow → `Settings` → `API`) | Sí, con `DATASET_SOURCE = "roboflow"` |
| `CAMARA_URL` | Dirección de la cámara para el modo en vivo: la del túnel de Cloudflare (`https://….trycloudflare.com`) | No (sección 9) |
| `CAMARA_TOKEN` | Token que imprime `puente_camara.py` en la PC de la cámara | Con el puente (sección 9) |
| `CF_ACCESS_CLIENT_ID` y `CF_ACCESS_CLIENT_SECRET` | Token de servicio, si el túnel está protegido con Cloudflare Access | No (sección 9) |

En Colab: panel izquierdo → 🔑 **Secrets** → agrega cada nombre con su valor y activa el acceso desde el notebook.

> ⚠️ Una versión anterior de este notebook tenía dos llaves de Roboflow escritas en celdas, así que quedaron guardadas en ese archivo y en cualquier copia que se haya compartido. **Revócalas en Roboflow (`Settings` → `API`), genera una nueva** y usa solo el *Secret*.

In [ ]:
import os


def get_secret(name: str, required: bool = False):
    """Lee un secreto de Colab (panel 🔑) o, fuera de Colab, de una variable de entorno."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except ImportError:
        value = os.environ.get(name)
    except Exception:  # el secreto no existe o el notebook no tiene acceso
        value = None
    if required and not value:
        raise RuntimeError(f"Falta el secreto {name}: agrégalo en el panel 🔑 Secrets de Colab.")
    return value

In [ ]:
import csv
import copy
import gc
import hashlib
import json
import random
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from importlib.metadata import version
from math import comb
from pathlib import Path

import cv2
import Levenshtein
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import yaml
from IPython.display import Image as IPyImage, Video, display
from PIL import Image, ImageDraw, ImageFont
from supervision.metrics import MeanAveragePrecision
from tqdm.auto import tqdm
from trackers import OCSORTTracker
from ultralytics import YOLO
from ultralytics.data.utils import IMG_FORMATS
from ultralytics.utils.torch_utils import get_flops, get_num_params

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
FP16 = 16 if CUDA else None  # inferencia en FP16 en la GPU (argumento `quantize` de Ultralytics 8.4)

print("GPU:", torch.cuda.get_device_name(0) if CUDA else "no disponible: todo será muy lento")
for package in ["torch", "ultralytics", "trackers", "supervision", "fast-plate-ocr", "onnxruntime"]:
    print(f"{package:<16}{version(package)}")

### Carpeta de trabajo en Google Drive
Colab borra `/content` al cerrar la sesión. El clon del repositorio contiene las entradas versionadas y recibe los resultados publicables; los entrenamientos y archivos temporales se guardan en `/content/SIRAP_workspace` durante la sesión.

La celda siguiente monta Drive, define **todas** las rutas de entrada en un solo lugar y al final muestra cuáles encontró (`ok`, `vacío` o `no está`). Las que faltan no detienen el notebook: la sección que usa cada una se salta.

| Archivo del repositorio o carpeta temporal | Para qué | Sección |
|---|---|---|
| `savh_ground_truth_subset.zip` | 739 recortes de SAVH Guayaquil con la lectura de la cámara en `ground_truth.csv` | 1.3 |
| Hoja de Google **Revisión placas SAVH (142 legibles)** | Correcciones de esa referencia (se lee con tu cuenta de Google) | 2.2 |
| `revision_placas_142.csv` | Copia en CSV de la hoja, solo si no se puede leer la hoja (Archivo → Descargar → CSV) | 2.2 |
| `authorized_plates.csv` | Placas autorizadas fuera de la base de la planta (excepciones), una por línea | 3.5 y 6.3 |
| `video_camiones.mp4` | Video real del punto de control, con `USE_SAMPLE_VIDEO = False` | 6.1 |
| `video_ground_truth.csv` | Referencia anotada del video (placa e intervalo de cuadros) | 7.1 y 7.2 |
| `garita/imagenes/` y `garita/ground_truth.csv` | Imágenes completas de la garita con la placa transcrita | 7.3 |
| `dataset_placas_yolo.zip` | Dataset propio en formato YOLO, si `DATASET_SOURCE = "local"` | 1.1 |

`SAVH_Guayaquil_ANPR_Plate_Images_Dataset.zip` y `SAVH_YOLOv8CLS_Plate_Legibility_Dataset.zip.zip` también están en la carpeta, pero este notebook no los usa: el primero tiene las 2000 imágenes de las que sale el subconjunto, y el segundo es el dataset de legibilidad derivado de LPLCv2 (placas brasileñas). 

**Salidas.** Cada experimento escribe en `experimentos/<dataset>/`: `runs/` (entrenamientos y validaciones de Ultralytics), `resultados/` (métricas en JSON y CSV, y registros de decisiones), `figuras/` (gráficos para el informe) y `videos/` (videos anotados). La evaluación del OCR en SAVH va a `ocr/`; la base de la planta se actualiza en su propio archivo, con una copia del original en `respaldo_bd/`. El modo en vivo (sección 9) guarda en `en_vivo/` el script del puente para la PC de la cámara y, por cada sesión, la grabación, el registro de decisiones y el resumen.

**Para retomar en una sesión nueva:** corre la sección 0 completa (no reentrena: los pesos ya están en Drive) y después las celdas que pide la sección que necesites, según la tabla de la portada (**Cómo usarlo**).

In [ ]:
WORK_DIR = SIRAP_PATHS["workspace"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", WORK_DIR)

# ===================== Archivos de entrada (todas las rutas en un solo lugar) =====================
SAVH_ZIP_CANDIDATES = [REPO_ROOT / "datasets/ecuador/savh/revision_visual/savh_ground_truth_subset.zip", REPO_ROOT / "evaluation/ecuador/savh/artifacts/revision_visual/savh_ground_truth_subset.zip", WORK_DIR / "savh_ground_truth_subset.zip"]
REVIEW_SHEET_ID = "1l79GrAYGFkTiOYxgKDT1_BEi_b-6Duo1_QqUNlcv7yM"  # hoja «Revisión placas SAVH (142 legibles)»
REVIEW_CSV_FALLBACK = REPO_ROOT / "datasets/ecuador/savh/ocr/revision_placas_142_leida.csv"
AUTHORIZED_LIST_PATH = REPO_ROOT / "data/authorized_plates.csv"
REAL_VIDEO_PATH = WORK_DIR / "video_camiones.mp4"                 # video real del punto de control
VIDEO_GROUND_TRUTH_PATH = REPO_ROOT / "evaluation/ecuador/blind_evaluation/video_ground_truth.csv"     # columnas: plate_gt, frame_start, frame_end
E2E_IMAGES_DIR = REPO_ROOT / "datasets/ecuador/gate/imagenes"                 # imágenes completas de la garita (sección 7.3)
E2E_GROUND_TRUTH_CSV = REPO_ROOT / "datasets/ecuador/gate/ground_truth.csv"   # columnas: image_path, plate_gt
LOCAL_DATASET_ZIP = REPO_ROOT / "datasets/ecuador/yolo/dataset_placas_yolo.zip"          # solo con DATASET_SOURCE = "local"
LIVE_DIR = WORK_DIR / "en_vivo"                                   # modo en vivo (sección 9): puente, grabaciones y registros
BRIDGE_SCRIPT_PATH = LIVE_DIR / "puente_camara.py"                # script para la PC de la cámara (celda 9.1)
LIVE_DIR.mkdir(exist_ok=True)


# Entradas según la estructura actual del repositorio
SAVH_ZIP_CANDIDATES = [REPO_ROOT / "datasets/ecuador/savh/revision_visual/savh_ground_truth_subset.zip", REPO_ROOT / "evaluation/ecuador/savh/artifacts/revision_visual/savh_ground_truth_subset.zip"]
REVIEW_CSV_FALLBACK = REPO_ROOT / "datasets/ecuador/savh/ocr/revision_placas_142_leida.csv"
AUTHORIZED_LIST_PATH = REPO_ROOT / "data/authorized_plates.csv"
REAL_VIDEO_PATH = WORK_DIR / "video_camiones.mp4"
VIDEO_GROUND_TRUTH_PATH = REPO_ROOT / "evaluation/ecuador/blind_evaluation/video_ground_truth.csv"
E2E_IMAGES_DIR = REPO_ROOT / "datasets/ecuador/gate/imagenes"
E2E_GROUND_TRUTH_CSV = REPO_ROOT / "datasets/ecuador/gate/ground_truth.csv"
LOCAL_DATASET_ZIP = REPO_ROOT / "datasets/ecuador/yolo/dataset_placas_yolo.zip"


def input_status(path: Path) -> str:
    """Estado de una entrada: «ok», «vacío» (archivo de 0 bytes) o «no está»."""
    if not path.exists():
        return "no está"
    if path.is_file() and path.stat().st_size == 0:
        return "vacío"
    return "ok"


savh_zip_path = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), SAVH_ZIP_CANDIDATES[0])
display(pd.DataFrame(
    [{"entrada": label, "estado": input_status(path), "ruta": str(path)} for label, path in (
        ("SAVH: recortes y ground_truth.csv", savh_zip_path),
        ("Revisión de placas: copia CSV (opcional)", REVIEW_CSV_FALLBACK),
        ("Placas autorizadas (excepciones)", AUTHORIZED_LIST_PATH),
        ("Video real", REAL_VIDEO_PATH),
        ("Referencia del video", VIDEO_GROUND_TRUTH_PATH),
        ("Imágenes de la garita", E2E_IMAGES_DIR),
        ("Referencia de la garita", E2E_GROUND_TRUTH_CSV),
        ("Dataset propio (YOLO)", LOCAL_DATASET_ZIP),
    )]).set_index("entrada"))
print("La hoja de revisión se lee en la sección 2.2 con tu cuenta de Google (ID " + REVIEW_SHEET_ID + ").")

### Configuración

Todos los parámetros del experimento están en la celda siguiente, agrupados por tema:

- **Dataset:** origen (`DATASET_SOURCE`) y proyecto y versión de Roboflow.
- **Detector:** pesos iniciales, `batch` y argumentos de inferencia (`infer_kwargs`, vacío: YOLO11 usa el NMS por defecto de Ultralytics). `video_imgsz` es el tamaño al que el detector analiza el video, las imágenes de la garita y la cámara en vivo: 1280 px (la resolución de la cámara); la evaluación en *val* y *test* usa siempre 640, el tamaño de entrenamiento. `CONFIGS` tiene una entrada por configuración del detector, con el nombre que aparece en la base y en las tablas (y en el video, con `SHOW_VIDEO_TITLE = True`); la recorren las secciones 6, 7 y 9.
- **Entrenamiento:** preset, épocas, fracción de los datos, tamaño de imagen, paciencia del *early stopping* y semilla.
- **Umbrales:** piso del umbral calibrado y precisión mínima para decidir sin verificación manual; los umbrales mismos se calibran en la sección 4.1.
- **Seguimiento y OCR:** cuadros para confirmar un *track*, tamaños mínimos de placa para seguirla y para leerla, modelo de fast-plate-ocr y confianza mínima del OCR.
- **Control de acceso:** formatos válidos de placa.
- **Metas del proyecto:** mAP@0.5 ≥ 0,85, EMR ≥ 80 % y ≤ 1 s por imagen, que aparecen como líneas de referencia en los gráficos.
- **Carpetas de salida y utilidades** comunes: nombre de la corrida, cargar el detector y armar sus argumentos de inferencia (`detector_kwargs`), guardar y leer resultados en JSON.

**Protocolo de entrenamiento.** Dataset y splits fijos, tamaño de imagen 640, semilla 0, *early stopping* con paciencia 10 y lote efectivo de 64 imágenes: Ultralytics acumula gradientes hasta 64 imágenes por paso (`nbs=64`), así que `batch` solo cambia la memoria usada. Con este tamaño de dataset Ultralytics elige AdamW.

**Por qué YOLO11-s.** Es un detector pequeño (≈9,4 M de parámetros y ≈21 GFLOPs a 640 px; mAP@0.5:0.95 de 47,0 en COCO según Ultralytics) que corre rápido incluso en hardware modesto y se exporta a ONNX para correr en CPU dentro de la app. Las placas son una sola clase, mucho más simple que las 80 de COCO, y en *test* supera con holgura la meta de mAP@0.5 (sección 5.1). Otros tamaños: `yolo11n.pt`, más liviano, o `yolo11m.pt` y `yolo11l.pt`, más precisos en COCO.

**NMS.** YOLO11 propone varias cajas por objeto y se queda con la mejor mediante NMS (supresión de no máximos), con el umbral de solapamiento por defecto de Ultralytics (`iou=0.7`), igual al validar y al predecir. Las latencias de la sección 5.3 incluyen el NMS, en el postproceso.

**Presets.** `rapido` entrena con una muestra aleatoria fija del 30 % de *train* y valida cada época con el 25 % de *val*, en 15 épocas: sirve para probar el flujo completo y tener números preliminares. `completo` usa todo con 40 épocas y es el que va al informe. Mira cuánto tarda la primera época antes de dejarlo corriendo: con las 2 CPU de Colab la carga de datos suele ser el cuello de botella.

**Umbrales.** Los umbrales de confianza no se fijan a mano: se calibran sobre *val* en la sección 4.1. El estudio de sensibilidad de los hiperparámetros de entrenamiento está en la sección 4.2.


In [ ]:
# ===================== Dataset (detalles en la sección 1.1) =====================
DATASET_SOURCE = "roboflow"  # "roboflow" | "local"
RF_WORKSPACE = "roboflow-universe-projects"
RF_PROJECT = "license-plate-recognition-rxg4e"
RF_VERSION = 11
DATASET_ID = f"{RF_PROJECT}-v{RF_VERSION}" if DATASET_SOURCE == "roboflow" else LOCAL_DATASET_ZIP.stem

# ===================== Detector =====================
DETECTORS = {
    "YOLO11": {
        "loader": YOLO,
        "weights": "yolo11s.pt",         # yolo11n / s / m / l / x
        "batch": 16,
        "deterministic": True,
        "infer_kwargs": {},              # NMS por defecto de Ultralytics (iou=0.7): YOLO11 no tiene cabeza sin NMS
        "video_imgsz": 1280,             # video, garita y en vivo: la resolución de la cámara (a 640 no ve placas chicas)
    },
}
MODEL_COLORS = {"YOLO11": "#2a78d6"}  # color del modelo en todos los gráficos

# ===================== Configuraciones que corren el pipeline (secciones 6, 7 y 9) =====================
# etiqueta → (modelo, cambios en la inferencia, sufijo de los resultados). Una variante de inferencia
# (otros argumentos de predict) se agregaría aquí con su propio sufijo, sin tocar los resultados principales.
CONFIGS = {name: (name, None, "") for name in DETECTORS}

# ===================== Entrenamiento =====================
TRAINING_PRESET = "completo"  # "rapido" para probar el flujo · "completo" para el informe
PRESETS = {
    "rapido":   {"epochs": 15, "train_fraction": 0.30, "val_fraction": 0.25},
    "completo": {"epochs": 40, "train_fraction": 1.00, "val_fraction": 1.00},
}
EPOCHS = PRESETS[TRAINING_PRESET]["epochs"]
TRAIN_FRACTION = PRESETS[TRAINING_PRESET]["train_fraction"]
VAL_FRACTION = PRESETS[TRAINING_PRESET]["val_fraction"]
IMGSZ = 640
PATIENCE = 10          # early stopping: épocas seguidas sin mejorar en val
SEED = 0
WORKERS = 2            # Colab tiene 2 CPU
EVAL_BATCH = 16
FORCE_RETRAIN = False  # True borra el entrenamiento guardado de este preset y lo repite desde cero

# ===================== Umbrales de confianza (se calibran en la sección 4.1) =====================
CONF_FLOOR = 0.10                 # piso para el umbral calibrado
DECISION_PRECISION_TARGET = 0.95  # precisión mínima en val para aceptar una detección sin verificación manual

# ===================== Seguimiento y OCR (secciones 3.4 y 6) =====================
MINIMUM_CONSECUTIVE_FRAMES = 10
MIN_PLATE_AREA_RATIO = 0.001  # placas más chicas (fracción del frame) ni se siguen
OCR_PLATE_AREA_RATIO = 0.002  # tamaño mínimo para leer: en video_camiones.mp4 las placas de camiones detenidos miden 0,3–0,5 % del frame
OCR_MODEL_NAME = "cct-s-v2-global-model"  # fast-plate-ocr; "cct-xs-v2-global-model" es más chico y, en SAVH, algo menos preciso
OCR_CONF_MIN = None   # confianza mínima del OCR para decidir sin verificación manual; None = no se usa. 4.3 sugiere 0.99 (el de la app)

# ===================== Control de acceso (secciones 2.1 y 3.6) =====================
# Placas ecuatorianas: 3 letras + 3 o 4 dígitos. Con la revisión de placas, las 472 legibles de SAVH siguen uno
# de esos dos formatos. Lecturas con 2 letras + 4 o 5 dígitos solo aparecen por errores de la cámara en la
# referencia (GB01825 es GBO-1825): aceptarlas haría que normalize_plate las diera por buenas sin corregirlas.
PLATE_PATTERNS = [(3, 4), (3, 3)]  # (letras, dígitos), por frecuencia
# Para decidir, guardar y mostrar, las placas válidas se escriben XXX-0000 (format_plate, sección 2.1).

# ===================== Metas del proyecto (líneas de referencia en los gráficos) =====================
TARGET_MAP50 = 0.85
TARGET_EMR = 0.80
TARGET_E2E_S = 1.0

COLOR = sv.ColorPalette.from_hex([  # un color por track en los videos anotados
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# ===================== Carpetas de salida (separadas por dataset) =====================
EXPERIMENT_DIR = WORK_DIR / "experimentos" / DATASET_ID
RUNS_DIR = EXPERIMENT_DIR / "runs"           # entrenamientos y validaciones de Ultralytics
RESULTS_DIR = EXPERIMENT_DIR / "resultados"  # métricas por modelo (JSON / CSV) y registros de decisiones
FIGURES_DIR = EXPERIMENT_DIR / "figuras"     # gráficos (PNG) para el informe
VIDEOS_DIR = EXPERIMENT_DIR / "videos"       # videos anotados
OCR_RESULTS_DIR = SIRAP_PATHS["savh_ocr"] / "ocr_experiments"
    OCR_RESULTS_DIR.mkdir(parents=True, exist_ok=True)           # evaluación del OCR en SAVH (no depende del detector)
for folder in (RUNS_DIR, RESULTS_DIR, FIGURES_DIR, VIDEOS_DIR, OCR_RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Salidas de este experimento:", EXPERIMENT_DIR)


# ===================== Utilidades comunes =====================
def run_name(name: str) -> str:
    """Identificador de la corrida: modelo + pesos + preset, p. ej. 'YOLO11_yolo11s_rapido'."""
    return f"{name}_{Path(DETECTORS[name]['weights']).stem}_{TRAINING_PRESET}"


def best_weights_path(name: str) -> Path:
    """Prefiere la corrida local; si no existe, usa los pesos oficiales de Ecuador versionados."""
    local = RUNS_DIR / run_name(name) / "weights" / "best.pt"
    published = SIRAP_PATHS["detector_weights"]
    return local if local.exists() or not (name == "YOLO11" and published.exists()) else published


def trained_models() -> list:
    """Modelos de DETECTORS que ya tienen best.pt en este preset (avisa cuáles faltan)."""
    ready = [name for name in DETECTORS if best_weights_path(name).exists()]
    missing = [name for name in DETECTORS if name not in ready]
    if missing:
        print("Aviso: todavía sin entrenar en este preset:", ", ".join(missing))
    return ready


def load_detector(name: str):
    """Carga el detector entrenado (best.pt) de la corrida."""
    return DETECTORS[name]["loader"](str(best_weights_path(name)))


def detector_kwargs(name: str, override=None) -> dict:
    """Argumentos de inferencia del modelo (`infer_kwargs` en DETECTORS). `override` los cambia para una variante de
    CONFIGS; un valor None vuelve al valor por defecto de Ultralytics."""
    kwargs = {**DETECTORS[name]["infer_kwargs"], **(override or {})}
    return {key: value for key, value in kwargs.items() if value is not None}


def video_imgsz(name: str) -> int:
    """Tamaño de inferencia en el video, la garita y el modo en vivo (`video_imgsz` en DETECTORS; None = IMGSZ). La
    calibración y la evaluación en val y test (secciones 4.1 y 5) usan siempre IMGSZ, el tamaño de entrenamiento."""
    return DETECTORS[name].get("video_imgsz") or IMGSZ


def _to_json(value):
    """Convierte tipos de numpy y rutas a tipos que json.dumps sabe guardar."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"No se puede guardar {type(value)} en JSON")


def save_result(name: str, kind: str, data: dict) -> None:
    """Guarda un resultado del modelo en resultados/<corrida>_<kind>.json (lo leen otras secciones y la 8)."""
    (RESULTS_DIR / f"{run_name(name)}_{kind}.json").write_text(json.dumps(data, indent=2, default=_to_json))


def load_result(name: str, kind: str):
    """Lee resultados/<corrida>_<kind>.json, o None si todavía no existe."""
    path = RESULTS_DIR / f"{run_name(name)}_{kind}.json"
    return json.loads(path.read_text()) if path.exists() else None


def free_gpu() -> None:
    """Libera memoria de la GPU entre evaluaciones largas."""
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()


def to_detections(result) -> sv.Detections:
    """Resultado de Ultralytics → sv.Detections en float32 (en FP16 las cajas llegan en float16)."""
    detections = sv.Detections.from_ultralytics(result)
    detections.xyxy = detections.xyxy.astype(np.float32)
    if detections.confidence is not None:
        detections.confidence = detections.confidence.astype(np.float32)
    return detections

In [ ]:
# Salidas persistentes del repositorio; los checkpoints de entrenamiento quedan en /content
EXPERIMENT_DIR = WORK_DIR / "experimentos" / DATASET_ID
RUNS_DIR = EXPERIMENT_DIR / "runs"
RESULTS_DIR = SIRAP_PATHS["metrics"]
FIGURES_DIR = SIRAP_PATHS["figures"]
VIDEOS_DIR = SIRAP_PATHS["video_results"]
OCR_RESULTS_DIR = SIRAP_PATHS["savh_ocr"] / "ocr_experiments"
    OCR_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for folder in (RUNS_DIR, RESULTS_DIR, FIGURES_DIR, VIDEOS_DIR, OCR_RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

## 1. Datos y análisis exploratorio

El sistema usa dos conjuntos de datos con fines distintos: un dataset público de **detección** (imágenes con la caja de cada placa) para entrenar y evaluar YOLO11-s, y recortes reales de placas ecuatorianas de **SAVH Guayaquil** para evaluar la lectura del OCR. El video y las imágenes de la garita se usan en las secciones 6 y 7.

### 1.1 Dataset de detección de placas

El detector se entrena y evalúa con un dataset fijo y sus splits. Hay dos fuentes posibles:

- `DATASET_SOURCE = "roboflow"` (por defecto): **License Plate Recognition** de Roboflow Universe (`roboflow-universe-projects/license-plate-recognition-rxg4e`, versión 11: 10 125 imágenes — 7 057 *train*, 2 048 *val*, 1 020 *test* —, una clase, licencia CC BY 4.0). No es específico de Ecuador: sirve para aprender a detectar placas en general. En la celda de configuración puedes cambiar `RF_WORKSPACE`/`RF_PROJECT`/`RF_VERSION` por un proyecto propio.
- `DATASET_SOURCE = "local"`: un `.zip` propio en formato YOLO (`train/`, `valid/`, `test/` con `images/` y `labels/`, más `data.yaml`) en `datasets/ecuador/savh/`, por ejemplo las imágenes de la garita anotadas en Roboflow y exportadas en formato YOLO.

**Para el informe:** lo ideal es que `test` tenga imágenes del dominio real (camiones en la garita). Un mAP alto en un test genérico no garantiza lo mismo con placas ecuatorianas.

La celda copia el dataset al disco local de Colab, reescribe `data.yaml` con rutas absolutas (el de Roboflow trae rutas `../train/images` que no siempre se resuelven bien) y, en el preset `rapido`, crea las listas con la muestra aleatoria fija. Imprime la carpeta del dataset y sus clases.


In [ ]:
# 1.1 · Dataset de detección: descarga (o .zip propio), data.yaml con rutas absolutas y muestra del preset
# Disco local de Colab: leer miles de imágenes desde Drive en cada época sería mucho más lento.
DATASET_DIR = Path("/content/datasets") / DATASET_ID

# Obtener la clave de API de Roboflow desde los Secretos de Colab
ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")

def get_dataset() -> Path:
    """Descarga o descomprime el dataset (una sola vez por sesión) y devuelve la carpeta con data.yaml."""
    if not list(DATASET_DIR.rglob("data.yaml")):
        if DATASET_SOURCE == "roboflow":
            if not ROBOFLOW_API_KEY:
                raise RuntimeError("Falta el secreto ROBOFLOW_API_KEY (sección 0).")
            from roboflow import Roboflow
            DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
            project = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT)
            # "yolov8" es el nombre del formato de exportación de Roboflow: el mismo formato de etiquetas YOLO que usa YOLO11
            project.version(RF_VERSION).download("yolov8", location=str(DATASET_DIR), overwrite=True)
        elif DATASET_SOURCE == "local":
            with zipfile.ZipFile(LOCAL_DATASET_ZIP) as archive:
                archive.extractall(DATASET_DIR)
        else:
            raise ValueError("DATASET_SOURCE debe ser 'roboflow' o 'local'")
    return min(DATASET_DIR.rglob("data.yaml"), key=lambda p: len(p.parts)).parent


def write_data_yaml(root: Path) -> Path:
    """data.yaml con rutas absolutas y los splits que existan (Roboflow llama 'valid' al de validación)."""
    original = yaml.safe_load((root / "data.yaml").read_text())
    names = original.get("names", ["placa"])
    if isinstance(names, list):
        names = dict(enumerate(names))
    data = {"path": str(root.resolve()), "names": names}
    for key, candidates in {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}.items():
        folder = next((c for c in candidates if (root / c / "images").is_dir()), None)
        if folder:
            data[key] = f"{folder}/images"
    if "test" not in data:
        print("Aviso: el dataset no trae split 'test'; se evaluará sobre val (no es lo ideal).")
        data["test"] = data["val"]
    if len(names) > 1:
        print(f"Aviso: el dataset tiene {len(names)} clases; el pipeline asume que todas son placas.")
    out = root / "data_abs.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


def list_images(folder: Path) -> list:
    """Imágenes de una carpeta (extensiones que acepta Ultralytics), en orden."""
    return sorted(p for p in Path(folder).iterdir() if p.suffix[1:].lower() in IMG_FORMATS)


def write_subset_yaml(data_yaml: Path) -> Path:
    """Preset 'rapido': muestra aleatoria fija de train y val, la misma en cada corrida."""
    if TRAIN_FRACTION >= 1 and VAL_FRACTION >= 1:
        return data_yaml
    data = yaml.safe_load(data_yaml.read_text())
    root, rng = Path(data["path"]), random.Random(SEED)
    for key, fraction in (("train", TRAIN_FRACTION), ("val", VAL_FRACTION)):
        images = list_images(root / data[key])
        chosen = sorted(rng.sample(images, max(1, round(len(images) * fraction))))
        listing = root / f"{key}_{TRAINING_PRESET}.txt"
        listing.write_text("\n".join(str(p) for p in chosen) + "\n")
        data[key] = str(listing)
    out = root / f"data_{TRAINING_PRESET}.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


DATASET_ROOT = get_dataset()
DATA_YAML = write_data_yaml(DATASET_ROOT)       # evaluación: val y test completos
TRAIN_DATA_YAML = write_subset_yaml(DATA_YAML)  # entrenamiento: muestra fija en el preset 'rapido'
DATA = yaml.safe_load(DATA_YAML.read_text())
print("Dataset:", DATASET_ROOT, "| clases:", DATA["names"])

### 1.3 Datos de lectura: SAVH Guayaquil

`savh_ground_truth_subset.zip`, en `datasets/ecuador/savh/`, tiene 739 recortes de placa de SAVH Guayaquil: 472 en `placa_legible/` y 267 en `placa_no_legible/`, copiados sin cambios de las 2000 imágenes del dataset público (capturas del 7, 8 y 11 de junio de 2026). `ground_truth.csv` toma la placa del nombre de cada archivo.

**Esa placa no es una transcripción manual: es la lectura automática de la cámara Dahua.** Lo muestran sus errores: 142 de las 472 placas legibles (30 %) no siguen el formato ecuatoriano, y en la imagen la placa se lee bien (`GSO-7335` aparece como `GS07335`, `GTN-8737` como `6TN8737`, `PBT-2437` como `JPBT2437`). Además, 73 imágenes son copias idénticas de otras (43 grupos; quedan 666 únicas, 399 de ellas legibles) y las 267 no legibles también traen texto, que no se puede comprobar mirando la imagen.

Por eso SAVH se usa en varios pasos del notebook:

1. **Extraer el subconjunto** (la celda siguiente).
2. **Armar la referencia corregida** con la revisión de placas y marcar los duplicados (sección 2.2).
3. **Leer cada imagen única con fast-plate-ocr** y estudiar la confianza del OCR (sección 4.3).
4. **Calcular EMR y CER** sobre las legibles únicas, por estado de la referencia, con la ablación de la normalización (sección 5.4).
5. **Evaluar la regla de decisión** con una lista simulada de placas autorizadas (sección 5.5).

`SAVH_SAMPLE_SIZE` permite probar con una muestra chica de legibles. Si no encuentra el `.zip`, todo lo de SAVH se salta sin romper el resto del notebook.


In [ ]:
# 1.3 · Extrae los recortes de SAVH Guayaquil (una vez por sesión, en el disco local de Colab)
_missing = [name for name in ("WORK_DIR", "OCR_RESULTS_DIR", "SAVH_ZIP_CANDIDATES") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, "
                       "corre primero la sección 0 completa.")

SAVH_DIR = Path("/content/savh_ground_truth_subset")  # disco local de Colab: se extrae en cada sesión
SAVH_SAMPLE_SIZE = None  # p. ej. 50 para una prueba rápida con legibles; None = todas las imágenes únicas (4.3 y 5.4)

savh_zip = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), None)
if not (SAVH_DIR / "ground_truth.csv").exists() and savh_zip:
    with zipfile.ZipFile(savh_zip) as archive:
        archive.extractall(SAVH_DIR)

savh_gt = None
if (SAVH_DIR / "ground_truth.csv").exists():
    savh_gt = pd.read_csv(SAVH_DIR / "ground_truth.csv", dtype=str, keep_default_na=False)
    savh_gt.insert(0, "fila_gt", range(1, len(savh_gt) + 1))  # fila de datos: la misma numeración de la hoja de revisión
    print(f"{len(savh_gt)} recortes en {SAVH_DIR}")
    display(savh_gt["folder"].value_counts())
else:
    print("Aviso: no se encontró savh_ground_truth_subset.zip en el repositorio ni en `/content`; se salta SAVH.")

## 2. Preprocesamiento

**Imágenes del detector.** El preprocesamiento de las imágenes de detección ya quedó hecho en 1.1 (copia local del dataset, `data.yaml` con rutas absolutas y muestra fija del preset `rapido`); el resto lo aplica Ultralytics al entrenar e inferir: *letterbox* a 640 px conservando la proporción y los aumentos por defecto (mosaico, cambios de color HSV, volteo horizontal, escala y traslación). Antes del OCR, cada placa se recorta con un margen de 10 px (`crop_plate`, sección 3.4).

**Texto de las placas.** Esta sección prepara lo que se compara con el OCR: la normalización de cualquier lectura al formato ecuatoriano (2.1) y la referencia corregida de SAVH (2.2).

### 2.1 Normalización de placas
El texto de un lector de placas no siempre es una placa válida. Antes de decidir el acceso hace falta encontrar la placa en el texto, corregir confusiones típicas por posición y validar el formato. fast-plate-ocr ya devuelve solo letras y dígitos, casi siempre con forma de placa: en SAVH, 398 de sus 399 lecturas de legibles ya tenían un formato válido y la normalización corrigió una sola (`GP1226` → `GPI226`). La misma normalización se aplica también a la lectura de la cámara de SAVH, que sí trae errores de formato, y a lo que escribe a mano el operador en la app.

Las placas ecuatorianas tienen **3 letras + 3 o 4 dígitos** ([El Universo](https://www.eluniverso.com/noticias/2020/06/26/nota/7885368/placas-carros-como-identificarlas/)). En la referencia de SAVH aparecen también lecturas con 2 letras + 4 o 5 dígitos, pero salen de errores de la cámara (`GB01825` es `GBO-1825`), así que solo se aceptan los dos formatos reales.

`normalize_plate` hace tres pasos:

1. **Busca la placa dentro del texto.** Corta el texto en tramos (por espacios, guiones, saltos de línea y otra puntuación), descarta la leyenda `ECUADOR` impresa sobre la placa si viene en el texto, y prueba cada tramo y las uniones de 2 o 3 tramos seguidos (`GBA 1234`).
2. **Corrige por posición.** En las 3 primeras posiciones (letras) cambia un dígito por la letra parecida (0→O, 1→I, 5→S, 6→G, 8→B) y en las siguientes (dígitos) una letra por el dígito parecido (O→0, D→0, Q→0, I→1, Z→2, B→8); `|` se lee como 1 y `$` como S. Gana el candidato que valida con menos cambios, y se aceptan como máximo 2: en SAVH, ninguna lectura que necesitó 3 era correcta.
3. **Valida el formato.** Si ningún candidato valida, la lectura queda inválida y la regla de decisión la manda a verificación manual.

**Formato para decidir, guardar y mostrar.** `format_plate` escribe cada placa válida como en Ecuador: 3 letras, guion y 4 dígitos (`GBA-1234`). A las placas antiguas de 3 dígitos les agrega un 0 al inicio de los números (`GMS-429` → `GMS-0429`), para que todas queden en el formato XXX-0000; por eso `GMS-429` y `GMS-0429` cuentan como la misma placa. Ese es el formato de la lista de autorizados, de la regla de decisión, de la base de datos de la planta, de los registros y del video. La evaluación del OCR con SAVH compara las placas sin guion, porque mide la lectura y no el formato.

La celda termina con casos de prueba (lecturas reales de la cámara de SAVH y textos con la leyenda, separadores y símbolos) que deben pasar antes de seguir.


In [ ]:
# 2.1 · Normalización de placas al formato ecuatoriano
CONFUSIONS_TO_LETTER = {"0": "O", "1": "I", "5": "S", "6": "G", "8": "B"}  # dígito leído donde va una letra
CONFUSIONS_TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "Z": "2", "B": "8"}  # letra leída donde va un dígito
SYMBOLS_TO_CHAR = {"|": "1", "$": "S"}  # símbolos que un lector de texto libre puede escribir en lugar de un carácter
SEPARATORS = " \t\r\n-‐‑‒–—_.·•:,;+\"'`*()[]{}<>"  # cortan el texto; cualquier otro símbolo invalida su tramo
MAX_CORRECTIONS = 2       # en SAVH, ninguna lectura que necesitó 3 cambios era correcta
PLATE_HEADER = "ECUADOR"  # leyenda impresa sobre la placa: se descarta si un lector la transcribe junto con la placa
_SEGMENT = re.compile("[^" + re.escape(SEPARATORS) + "]+")  # tramo: caracteres seguidos sin separadores


def clean_plate_text(text) -> str:
    """Mayúsculas y solo letras y dígitos: 'gba-1234' → 'GBA1234'."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


def is_plate_header(segment: str) -> bool:
    """'ECUADOR' y sus lecturas aproximadas ('EQUADOR', 'ECUNDERG'): solo letras y a 3 cambios o menos."""
    return (len(segment) >= 5 and segment.isascii() and segment.isalpha()
            and Levenshtein.distance(segment, PLATE_HEADER) <= 3)


def as_plate(candidate: str, corrections: bool = True):
    """(placa, cambios) si el candidato tiene formato de placa, corrigiendo por posición si hace falta; si no, None."""
    for n_letters, n_digits in PLATE_PATTERNS:
        if len(candidate) != n_letters + n_digits:
            continue
        letters, digits = candidate[:n_letters], candidate[n_letters:]
        if corrections:
            letters = "".join(CONFUSIONS_TO_LETTER.get(c, c) for c in letters)
            digits = "".join(CONFUSIONS_TO_DIGIT.get(c, c) for c in digits)
        plate = letters + digits
        changes = sum(a != b for a, b in zip(plate, candidate))
        if changes <= MAX_CORRECTIONS and re.fullmatch(f"[A-Z]{{{n_letters}}}[0-9]{{{n_digits}}}", plate):
            return plate, changes
    return None


def find_plate(raw_text, search: bool = True, corrections: bool = True) -> tuple:
    """Busca la placa en el texto del OCR → (placa, formato válido, cambios, (inicio, fin) en el texto).

    Prueba cada tramo entre separadores y las uniones de 2 o 3 tramos seguidos ('GBA 1234'), sin la leyenda
    ECUADOR. Gana el candidato que valida con menos cambios y, a igualdad, el que aparece primero. Si ninguno
    valida, devuelve el texto limpio sin la leyenda, marcado como inválido. `search=False` y `corrections=False`
    apagan cada paso, para la ablación."""
    text = str(raw_text).upper()
    if not search:  # el texto entero como un solo candidato
        found = as_plate(clean_plate_text(text), corrections)
        return (found[0], True, found[1], None) if found else (clean_plate_text(text), False, 0, None)
    segments = [(m.start(), m.end(), "".join(SYMBOLS_TO_CHAR.get(c, c) for c in m.group()))
                for m in _SEGMENT.finditer(text)]
    segments = [s for s in segments if not is_plate_header(s[2])]
    best = None
    for i in range(len(segments)):
        for j in range(i + 1, min(i + 3, len(segments)) + 1):
            found = as_plate("".join(s[2] for s in segments[i:j]), corrections)
            if found and (best is None or found[1] < best[2]):
                best = (found[0], True, found[1], (segments[i][0], segments[j - 1][1]))
    return best or (clean_plate_text("".join(s[2] for s in segments)), False, 0, None)


def normalize_plate(raw_text: str) -> tuple[str, bool]:
    """Texto del OCR → (placa normalizada, formato válido)."""
    plate, is_valid, _, _ = find_plate(raw_text)
    return plate, is_valid


def format_plate(text) -> str:
    """Formato de Ecuador para decidir, guardar y mostrar: 3 letras, guion y 4 dígitos ('gba1234' → 'GBA-1234'). A las
    placas antiguas de 3 dígitos se les agrega un 0 al inicio de los números ('GMS-429' → 'GMS-0429'). Un texto que no
    es una placa válida queda limpio (mayúsculas, solo letras y dígitos), sin guion."""
    plate = clean_plate_text(text)
    match = re.fullmatch(r"([A-Z]{3})(\d{3,4})", plate)
    return f"{match.group(1)}-{match.group(2).zfill(4)}" if match else plate


def plate_confidence(text: str, char_probs) -> float:
    """Confianza del OCR en la placa: la menor probabilidad entre sus letras y dígitos dentro del texto
    (sin placa válida, entre todas las letras y dígitos). NaN si no hay probabilidades."""
    if char_probs is None or len(char_probs) != len(text):
        return float("nan")
    span = find_plate(text)[3]
    start, end = span if span and len(text.upper()) == len(text) else (0, len(text))
    probs = [p for c, p in zip(text[start:end], char_probs[start:end]) if c.isalnum() or c in SYMBOLS_TO_CHAR]
    return float(min(probs)) if probs else 0.0


# Casos de prueba: lecturas reales de la cámara en SAVH (revisión de placas) y textos con la leyenda ECUADOR,
# separadores y símbolos. Cada uno debe dar la placa esperada.
for raw, expected in [("GB01825", "GBO1825"), ("GS11034", "GSI1034"), ("6TK6566", "GTK6566"), ("GTE5D45", "GTE5045"),
                      ("GP1986", "GPI986"), ("gba-1234", "GBA1234"), ("ECUADOR\nGMS-429", "GMS429"),
                      ("EQUIDOR\nGTE-7040", "GTE7040"), ("ECUNDERG\nGB0-6445", "GBO6445"), ("G5V-3233", "GSV3233"),
                      ("G50-6164", "GSO6164"), ("G|G-430", "GIG430"), ("GSH+2315", "GSH2315"), ("GTT·4568", "GTT4568"),
                      ("ECUADOR GMS 429", "GMS429")]:
    normalized, is_valid = normalize_plate(raw)
    assert is_valid and normalized == expected, f"{raw!r} → {normalized} (se esperaba {expected})"
# Lecturas que deben quedar inválidas (→ verificación manual) en vez de convertirse en otra placa válida.
for raw in ["GTT-187G", "GPA-1G88", "OBX=660", "GBΦ6184", "115220", "ECUADOR", ""]:
    normalized, is_valid = normalize_plate(raw)
    assert not is_valid, f"{raw!r} no debería validar (salió {normalized})"
assert plate_confidence("ECUADOR\nGMS-429", [0.2] * 8 + [0.9, 0.8, 0.95, 0.3, 0.99, 0.7, 0.9]) == 0.7
for raw, expected in [("gba1234", "GBA-1234"), ("GBA-1234", "GBA-1234"), ("GMS429", "GMS-0429"), ("gms-429", "GMS-0429"),
                      ("PBA 123", "PBA-0123"), ("PBA-0123", "PBA-0123"), ("G8A1234", "G8A1234"), ("", "")]:
    assert format_plate(raw) == expected, f"{raw!r} → {format_plate(raw)!r} (se esperaba {expected!r})"
print("normalize_plate y format_plate: los casos de prueba pasan.")

### 2.2 Referencia corregida de SAVH (revisión de placas)

La hoja de Google **Revisión placas SAVH (142 legibles)**, en `datasets/ecuador/savh/`, tiene una fila por cada placa legible cuya lectura de la cámara no sigue el formato ecuatoriano. La columna `sugerencia` la propuso Claude el 16/10/2026 con reglas de formato y una revisión visual de cada imagen: 118 sugerencias coinciden con la imagen, 15 se corrigieron mirándola (la regla sola fallaba, por ejemplo `BX4948` → `PBX4948`) y 9 quedaron dudosas (filas con `n` = 7–9, 20–22 y 71–73 en la hoja). Son propuestas: la placa definitiva es la que confirme una persona mirando la imagen.

**Para confirmar**, en la última columna de la hoja se escribe una de estas respuestas. Basta con una fila por grupo de copias idénticas (columna `igual_a`): la respuesta se aplica a todo el grupo.

| Respuesta en la hoja | Placa que se usa | `estado_gt` |
|---|---|---|
| `ok` | la sugerencia | `confirmada` |
| una placa, p. ej. `GBU-1825` | esa placa | `corregida_por_revisor` |
| `ilegible` | ninguna: se excluye del EMR | `excluida_por_revisor` |
| vacía | la sugerencia, por ahora | `sugerida_sin_confirmar` |

Las filas que se agregaron a la hoja por desacuerdos entre la cámara y el OCR no traen sugerencia: ahí `ok` confirma la lectura de la cámara (`texto_actual`), y mientras no tengan respuesta la referencia sigue siendo esa lectura.

**Ojo con la O y la Q en la tercera letra.** La cámara escribe 0 para las dos y las sugerencias siempre proponen O, pero en la fila con `n` = 74 fast-plate-ocr leyó Q, y la misma placa de la fila 83 aparece en otras dos imágenes como `GSQ8533` según la cámara. Conviene mirar con cuidado las filas 74 y 83–85.

Las demás legibles quedan como `camara_formato_valido` (tienen formato válido y, en una muestra de 16, 15 coincidían con la imagen y 1 era dudosa) y las no legibles como `no_legible`. La hoja se lee con tu cuenta de Google (Colab pide permiso la primera vez); si no se puede, se usa `revision_placas_142.csv` de la clon del repositorio. La referencia corregida se guarda en `ocr/savh_ground_truth_corregido.csv`, junto con una copia de la hoja tal como se leyó, para que el informe cite datos fijos. Las copias idénticas se marcan para contar cada imagen una vez. Las dos celdas siguientes arman la referencia y muestran las imágenes que todavía no tienen respuesta.

Si la celda se corre sola en una sesión nueva, avisa qué celdas faltan: la sección 0, la extracción de SAVH (1.3) y la normalización (2.1).


In [ ]:
_missing = [name for name in ("OCR_RESULTS_DIR", "REVIEW_SHEET_ID", "REVIEW_CSV_FALLBACK", "PLATE_PATTERNS",
                              "clean_plate_text", "savh_gt", "SAVH_DIR") if name not in globals()]
if _missing:
    raise RuntimeError("Faltan variables de celdas anteriores: " + ", ".join(_missing) + ". En una sesión nueva, corre "
                       "primero la sección 0 completa, la extracción de SAVH (1.3) y la normalización (2.1).")

# 2.2 · Referencia corregida de SAVH
REVIEW_PLATE_COLUMN = "revision: ok o placa correcta"
REVIEW_OK = {"ok", "si", "sí"}
REVIEW_EXCLUDE = {"ilegible", "no legible", "excluir"}
CORRECTED_GT_PATH = OCR_RESULTS_DIR / "savh_ground_truth_corregido.csv"
REVIEW_SNAPSHOT_PATH = OCR_RESULTS_DIR / "revision_placas_142_leida.csv"  # la hoja tal como se leyó
STATE_PRIORITY = ["corregida_por_revisor", "excluida_por_revisor", "confirmada", "sugerida_sin_confirmar",
                  "camara_formato_valido", "no_legible"]  # dentro de un grupo de copias gana la respuesta más informativa


def has_plate_format(text: str) -> bool:
    """True si el texto (sin guion) es 3 letras + 3 o 4 dígitos (PLATE_PATTERNS)."""
    return any(len(text) == letters + digits and text[:letters].isalpha() and text[letters:].isdigit()
               for letters, digits in PLATE_PATTERNS)


def read_review_sheet() -> pd.DataFrame:
    """La hoja de revisión, leída con la cuenta de Google de Colab o, si no se puede, desde la copia en CSV."""
    try:
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        credentials, _ = default()
        values = gspread.authorize(credentials).open_by_key(REVIEW_SHEET_ID).sheet1.get_all_values()
        review = pd.DataFrame(values[1:], columns=values[0])
        print(f"Hoja de revisión leída desde Google Sheets: {len(review)} filas.")
    except Exception as error:  # fuera de Colab, sin permiso o sin conexión
        if not REVIEW_CSV_FALLBACK.exists():
            raise FileNotFoundError(
                f"no se pudo leer la hoja ({type(error).__name__}) y no existe {REVIEW_CSV_FALLBACK}. "
                "Descárgala como CSV (Archivo → Descargar → CSV) con ese nombre en `datasets/ecuador/savh/`.") from error
        review = pd.read_csv(REVIEW_CSV_FALLBACK, dtype=str, keep_default_na=False)
        print(f"No se pudo leer la hoja ({type(error).__name__}); se usa {REVIEW_CSV_FALLBACK.name}: {len(review)} filas.")
    review = review.astype(str).apply(lambda column: column.str.strip())
    missing = {"n", "fila_gt", "imagen", "texto_actual", "sugerencia", "chequeo_visual", REVIEW_PLATE_COLUMN} - set(review.columns)
    if missing:
        raise ValueError(f"a la hoja de revisión le faltan columnas: {sorted(missing)}")
    return review


def build_corrected_reference(gt: pd.DataFrame, review: pd.DataFrame) -> tuple:
    """Aplica la revisión a ground_truth.csv (por fila_gt) y marca las copias idénticas de una misma imagen."""
    ref = gt.rename(columns={"plate_gt": "plate_camara"}).set_index("fila_gt")
    ref["plate_gt"] = ref["plate_camara"]
    ref["estado_gt"] = np.where(ref["folder"] == "legible", "camara_formato_valido", "no_legible")
    ref["n_hoja"], ref["chequeo_visual"], ref["respuesta"] = "", "", ""
    warnings = []
    for rec in review.to_dict("records"):
        row = int(rec["fila_gt"])
        if row not in ref.index or ref.at[row, "plate_camara"] != rec["texto_actual"]:
            warnings.append(f"fila_gt {row} ({rec['texto_actual']}) no coincide con ground_truth.csv; se ignora.")
            continue
        if Path(ref.at[row, "image_path"]).name != Path(rec["imagen"]).name:
            warnings.append(f"fila_gt {row}: el nombre de imagen de la hoja no coincide; se usa fila_gt.")
        answer = rec[REVIEW_PLATE_COLUMN].lower().strip(" .")
        suggestion = clean_plate_text(rec["sugerencia"]) or clean_plate_text(rec["texto_actual"])  # sin sugerencia: la cámara
        if answer in REVIEW_OK:
            plate, state = suggestion, "confirmada"
        elif answer in REVIEW_EXCLUDE:
            plate, state = "", "excluida_por_revisor"
        elif answer:
            plate, state = clean_plate_text(answer), "corregida_por_revisor"
        elif rec["sugerencia"]:
            plate, state = suggestion, "sugerida_sin_confirmar"
        else:  # agregada por desacuerdo con el OCR y todavía sin respuesta: sigue la lectura de la cámara
            plate, state = suggestion, "camara_formato_valido"
        if state != "excluida_por_revisor" and not has_plate_format(plate):
            warnings.append(f"fila_gt {row}: «{plate}» no tiene formato de placa ecuatoriana.")
        ref.loc[row, ["plate_gt", "estado_gt", "n_hoja", "chequeo_visual", "respuesta"]] = [
            plate, state, rec["n"], rec["chequeo_visual"], answer]
    ref = ref.reset_index()

    # Copias idénticas: cada grupo cuenta una vez y todo el grupo toma la respuesta más informativa.
    first_by_digest, group = {}, []
    for row, path in zip(ref["fila_gt"], ref["image_path"]):
        digest = hashlib.md5((SAVH_DIR / path).read_bytes()).hexdigest()
        group.append(first_by_digest.setdefault(digest, row))
    ref["duplicado_de_fila"] = pd.array([g if g != r else pd.NA for g, r in zip(group, ref["fila_gt"])], dtype="Int64")
    priority = ref["estado_gt"].map({state: i for i, state in enumerate(STATE_PRIORITY)})
    best = (ref.assign(grupo=group, prioridad=priority).sort_values(["grupo", "prioridad"])
            .drop_duplicates("grupo").set_index("grupo"))
    ref["plate_gt"] = [best.at[g, "plate_gt"] for g in group]
    ref["estado_gt"] = [best.at[g, "estado_gt"] for g in group]
    return ref, warnings


savh_ref = None
if savh_gt is not None:
    try:
        review = read_review_sheet()
    except (FileNotFoundError, ValueError) as error:
        print("Aviso: sin revisión de placas,", error)
        print("Se salta la evaluación de SAVH: con la lectura de la cámara como referencia, el EMR no sería confiable.")
    else:
        review.to_csv(REVIEW_SNAPSHOT_PATH, index=False)
        savh_ref, warnings = build_corrected_reference(savh_gt, review)
        for warning in warnings:
            print("Aviso:", warning)
        savh_ref.to_csv(CORRECTED_GT_PATH, index=False)
        unique = savh_ref["duplicado_de_fila"].isna()
        display(pd.crosstab(savh_ref["estado_gt"], unique.map({True: "imágenes únicas", False: "copias"}),
                            margins=True, margins_name="total"))
        excluded = savh_ref["estado_gt"] == "excluida_por_revisor"
        changed = int(((savh_ref["plate_gt"] != savh_ref["plate_camara"]) & unique & ~excluded).sum())
        print(f"{changed} imágenes únicas cambian de placa respecto a la cámara y {int((unique & excluded).sum())} "
              f"quedan excluidas por el revisor. Referencia guardada en {CORRECTED_GT_PATH}")

In [ ]:
# 2.2 · Imágenes de la hoja de revisión que todavía no tienen respuesta
SHOW_PENDING_REVIEW = True  # imágenes de la hoja que todavía no tienen respuesta, una por grupo de copias

if SHOW_PENDING_REVIEW and savh_ref is not None:
    unique = savh_ref["duplicado_de_fila"].isna()
    added = (savh_ref["estado_gt"] == "camara_formato_valido") & (savh_ref["n_hoja"] != "")  # desacuerdos sin respuesta
    pending = savh_ref[unique & ((savh_ref["estado_gt"] == "sugerida_sin_confirmar") | added)]
    if pending.empty:
        print("Todas las placas de la hoja de revisión tienen respuesta.")
    else:
        pending = pending.assign(orden=pd.to_numeric(pending["n_hoja"], errors="coerce")).sort_values("orden")
        print(f"{len(pending)} imágenes por confirmar en la hoja de revisión (el número es la columna n de la hoja; en rojo, "
              "las dudosas; en azul, los desacuerdos con el OCR, con la lectura de la cámara).")
        columns = 6
        rows = -(-len(pending) // columns)
        fig, axes = plt.subplots(rows, columns, figsize=(2.7 * columns, 1.7 * rows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, rec in zip(axes.flat, pending.itertuples(index=False)):
            ax.imshow(Image.open(SAVH_DIR / rec.image_path).convert("RGB"))
            doubtful = rec.chequeo_visual == "dudosa"
            disagreement = rec.estado_gt == "camara_formato_valido"
            tag = " · dudosa" if doubtful else (" · desacuerdo" if disagreement else "")
            ax.set_title(f"n {rec.n_hoja} · {rec.plate_gt}{tag}", fontsize=8.5,
                         color="#b91c1c" if doubtful else ("#1d4ed8" if disagreement else "#0b0b0b"))
        plt.tight_layout()
        plt.show()

## Publicar resultados en GitHub
Los archivos se guardan en las carpetas del clon siguiendo la estructura actual. Para enviarlos al repositorio, configura el secreto `GITHUB_TOKEN` en Colab con permiso `Contents: Read and write` y ejecuta la celda siguiente. El token no se guarda en el notebook.

In [ ]:
# Publicación explícita de resultados (no guardar el token aquí).
# Crea GITHUB_TOKEN en Colab > Secretos, con permiso Contents: Read and write.
PUBLISH_TO_GITHUB = True
if PUBLISH_TO_GITHUB:
    publish_artifacts(
        REPO_ROOT,
        ["results/metrics", "results/figures", "results/ecuador",
         "evaluation/ecuador/savh/artifacts/ocr", "models/ecuador"],
        "Save Ecuador notebook outputs",
    )
else:
    print("Resultados guardados en el clon local. Activa PUBLISH_TO_GITHUB para enviarlos a GitHub.")